# 🎯 Validar T-DEED (action spotting de video) sobre nuestros clips

Corre un spotter **pre-entrenado** (T-DEED, ganador SoccerNet Ball 2024) que detecta **Shot y Goal**
mirando el video. **Criterio de éxito:** encuentra el **gol de psg-inter** y **no inventa tiros en psg_bayern**.

> ⚠️ Es un repo de investigación — puede necesitar ajustes en dependencias/rutas/salida. Activá **GPU**.
> Notebook aparte del pipeline de tracking (deps distintas).


## 0. GPU


In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU — activá una')


## 1. Clonar T-DEED + instalar


In [ ]:
import os
REPO='/content/T-DEED'
if not os.path.exists(REPO):
    !git clone -q https://github.com/arturxe2/T-DEED.git {REPO}
%cd {REPO}
!pip install -q -r requirements.txt
!pip install -q gdown
print('T-DEED clonado en', REPO)


## 2. Bajar el checkpoint (SoccerNetBall_challenge1, 12 clases con Shot/Goal)

Descarga la carpeta de checkpoints de Google Drive (sin API). Puede tardar.


In [ ]:
# La carpeta pública de checkpoints de T-DEED:
!gdown -q --folder https://drive.google.com/drive/folders/1sxZalU_hCwL8ITZCU9VqSWE8dB94lJty -O /content/tdeed_ckpts || echo 'ojo: revisar descarga'
!echo '--- contenido descargado ---'; find /content/tdeed_ckpts -maxdepth 2 -type d 2>/dev/null | head -40
# T-DEED espera el checkpoint en ./checkpoints/<model_name>/ — lo enlazamos:
import os, glob, shutil
os.makedirs('checkpoints', exist_ok=True)
for d in glob.glob('/content/tdeed_ckpts/*'):
    name=os.path.basename(d)
    dst=os.path.join('checkpoints', name)
    if os.path.isdir(d) and not os.path.exists(dst): shutil.move(d, dst)
print('checkpoints disponibles:', os.listdir('checkpoints'))


## 3. Los dos clips

Subilos desde tu compu, o poné rutas de Drive. Necesitás **psg_bayern** (0 tiros reales) y **psg-inter** (1 gol real).


In [ ]:
from google.colab import files
print('Subí psg_bayern_720p.mp4 ...'); a=files.upload(); PSG_BAYERN=os.path.abspath(list(a.keys())[0])
print('Subí el clip psg-inter (mp4) ...'); b=files.upload(); PSG_INTER=os.path.abspath(list(b.keys())[0])
print('bayern:', PSG_BAYERN); print('inter :', PSG_INTER)


## 4. Inferencia sobre cada clip

Comando del repo: `inference.py --model SoccerNetBall_challenge1 --video_path X --frame_width 796 --frame_height 448 --inference_threshold 0.3`.
Si el `--model` no existe con ese nombre, revisá `os.listdir('checkpoints')` de arriba y usá el que aparezca.


In [ ]:
MODEL='SoccerNetBall_challenge1'   # ajustar al nombre real que apareció en checkpoints/
print('==================== psg_bayern (esperado: NINGÚN tiro) ====================')
!python3 inference.py --model {MODEL} --video_path "{PSG_BAYERN}" --frame_width 796 --frame_height 448 --inference_threshold 0.3
print('\n==================== psg-inter (esperado: ENCUENTRA el gol) ====================')
!python3 inference.py --model {MODEL} --video_path "{PSG_INTER}" --frame_width 796 --frame_height 448 --inference_threshold 0.3


## 5. Qué mirar

- En **psg-inter**: que aparezca una detección **Goal** (y/o Shot) cerca del minuto del gol.
- En **psg_bayern**: que **NO** dispare tiros (los que veían las reglas eran despejes).
- La salida de `inference.py` puede ser un print o un archivo — si no se entiende, pasámela y ajustamos el parseo.

Si T-DEED pasa este test, tenés **tiros/goles/faltas resueltos sin etiquetar nada**.
